# Verification Suite

An independent, from-scratch re-implementation that re-checks selected results (Table 1 size, Table 2
power, Table 7 HAC/fixed-b size, Table 9 BTC leg). Each check prints `[PASS]` or `[FAIL]` and the
notebook ends with a `Passed: N/N checks` summary.

A check passes when the new run lands within a stated tolerance (a few percentage points) of a
reference value recorded from an earlier run.
Run top to bottom in a fresh kernel.

## Setup & Core Functions

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats


### Constants

In [2]:
# Global constants (edit here to change experiment parameters)
SEED = 42
H, T = 90, 2000
REPS = 1000
ALPHA = 0.20

FAST_RUN = False
FAST_FACTOR = 20


def effective_reps(reps: int) -> int:
    """Scale `reps` down by FAST_FACTOR when FAST_RUN is on."""
    if not FAST_RUN:
        return reps
    return max(1, reps // FAST_FACTOR)


REPS = effective_reps(REPS)
rng = np.random.default_rng(SEED)
z90 = stats.norm.ppf(0.90)
PASS_LOG = []

DATA_DIR = Path('data')
BTC_CSV = DATA_DIR / 'btc_binance_daily.csv'

print(f'H={H} T={T} REPS={REPS} FAST_RUN={FAST_RUN}')


H=90 T=2000 REPS=1000 FAST_RUN=False


### Reference values

Recorded from an earlier run with seed 42 and kept inline as plain Python dicts.

In [3]:
REF = {
    'table1_size_normal': {
        'tolerance_pp': 3.0,
        's': {
            1: {'kupiec': 79.0, 'christoffersen': 100.0},
            30: {'kupiec': 14.9, 'christoffersen': 55.6},
            60: {'kupiec': 7.2, 'christoffersen': 5.5},
            90: {'kupiec': 5.4, 'christoffersen': 1.9},
        },
        'empirical_coverage_pct': 80.22,
        'empirical_coverage_tolerance_pp': 0.3,
    },
    'table2_power_unconditional': {
        'tolerance_pp': 4.0,
        'true_coverage': {
            0.70: {60: 30.2, 90: 23.0},
            0.90: {60: 32.6, 90: 24.0},
        },
    },
    'table2_power_clustering': {
        'tolerance_pp': 3.0,
        'coverage_pct': 80.2,
        'coverage_tolerance_pp': 0.5,
        's': {
            60: {'christoffersen': 4.7, 'kupiec': 6.8},
            90: {'christoffersen': 2.8, 'kupiec': 6.5},
        },
    },
    'table7_hac_fixed_b': {
        'tolerance_pp': 3.0,
        'T': {1000: 13.0, 2000: 11.2, 4000: None},
    },
    'garch_btc': {
        'coverage_pct': {'value': 90.1, 'tolerance_pp': 0.5},
        'kupiec_s30': {'value': 0.007, 'tolerance': 0.02},
        'kupiec_s90': {'value': 0.115, 'tolerance': 0.05},
        'christ_s90': {'value': 0.238, 'tolerance': 0.08},
    },
}


def check(label, got, expected, tol):
    ok = abs(got - expected) <= tol
    PASS_LOG.append((label, got, expected, ok))
    print(f"{'[PASS]' if ok else '[FAIL]'} {label}: got={got:.3f} reference={expected:.3f} (tol={tol})")
    return ok


print('OK')

OK


### Coverage-test statistics

In [4]:
def kupiec_stat(V, alpha=0.20):
    """Kupiec (1995) unconditional coverage test; p-value per replication."""
    n = V.shape[1]
    V1 = V.sum(1)
    ph = np.clip(V1 / n, 1e-10, 1 - 1e-10)
    LR = -2 * (V1 * np.log(alpha / ph) + (n - V1) * np.log((1 - alpha) / (1 - ph)))
    return 1 - stats.chi2.cdf(np.where(V1 == 0, 0., LR), 1)


def christ_stat(V, alpha=0.20):
    """Christoffersen (1998) conditional coverage test; p-value per replication."""
    a, b = V[:, :-1], V[:, 1:]
    n00 = ((a == 0) & (b == 0)).sum(1)
    n01 = ((a == 0) & (b == 1)).sum(1)
    n10 = ((a == 1) & (b == 0)).sum(1)
    n11 = ((a == 1) & (b == 1)).sum(1)
    p01 = n01 / np.maximum(n00 + n01, 1)
    p11 = n11 / np.maximum(n10 + n11, 1)
    ph = (n01 + n11) / np.maximum(n00 + n01 + n10 + n11, 1)
    eps = 1e-300
    lnum = (n00 * np.log(np.clip(1 - p01, eps, 1)) + n01 * np.log(np.clip(p01, eps, 1))
            + n10 * np.log(np.clip(1 - p11, eps, 1)) + n11 * np.log(np.clip(p11, eps, 1)))
    lden = ((n00 + n10) * np.log(np.clip(1 - ph, eps, 1)) + (n01 + n11) * np.log(np.clip(ph, eps, 1)))
    n = V.shape[1]
    V1 = V.sum(1)
    phu = np.clip(V1 / n, 1e-10, 1 - 1e-10)
    LRu = -2 * (V1 * np.log(alpha / phu) + (n - V1) * np.log((1 - alpha) / (1 - phu)))
    LRc = np.where(V1 == 0, 0., LRu) + 2 * (lnum - lden)
    return 1 - stats.chi2.cdf(LRc, 2)


def kupiec_scalar(v, alpha=0.20):
    """Kupiec test for a single 0/1 violation series (not a MC batch)."""
    V1 = int(v.sum())
    n = len(v)
    if V1 == 0:
        return 1.
    ph = V1 / n
    LR = -2 * (V1 * np.log(alpha / ph) + (n - V1) * np.log((1 - alpha) / (1 - ph)))
    return float(1 - stats.chi2.cdf(LR, 1))


def christ_scalar(v, alpha=0.20):
    """Christoffersen test for a single 0/1 violation series (not a MC batch)."""
    a = v[:-1]
    b = v[1:]
    n00 = ((a == 0) & (b == 0)).sum()
    n01 = ((a == 0) & (b == 1)).sum()
    n10 = ((a == 1) & (b == 0)).sum()
    n11 = ((a == 1) & (b == 1)).sum()
    p01 = n01 / max(n00 + n01, 1)
    p11 = n11 / max(n10 + n11, 1)
    ph = (n01 + n11) / max(len(b), 1)
    eps = 1e-300
    lnum = (n00 * np.log(max(1 - p01, eps)) + n01 * np.log(max(p01, eps))
            + n10 * np.log(max(1 - p11, eps)) + n11 * np.log(max(p11, eps)))
    lden = ((n00 + n10) * np.log(max(1 - ph, eps)) + (n01 + n11) * np.log(max(ph, eps)))
    V1 = int(v.sum())
    n = len(v)
    phu = min(max(V1 / n, 1e-10), 1 - 1e-10)
    if V1 > 0:
        LRu = -2 * (V1 * np.log(alpha / phu) + (n - V1) * np.log((1 - alpha) / (1 - phu)))
    else:
        LRu = 0.
    return float(1 - stats.chi2.cdf(LRu + 2 * (lnum - lden), 2))


def hac_z(V, H, alpha=0.20, L=None):
    """HAC z-statistic for the violation rate (Bartlett kernel, lag L, default H-1)."""
    n = V.shape[1]
    if L is None:
        L = H - 1
    L = min(L, n - 2)
    Vc = V - V.mean(axis=1, keepdims=True)
    lrv = (Vc * Vc).mean(axis=1)
    for k in range(1, L + 1):
        w = 1 - k / (L + 1)
        lrv += 2 * w * (Vc[:, k:] * Vc[:, :-k]).sum(axis=1) / n
    lrv = np.maximum(lrv, 1e-12)
    return (V.mean(axis=1) - alpha) / np.sqrt(lrv / n), L, n


def cv_fixed_b(b):
    """Fixed-b asymptotic critical value approximation."""
    return 1.9600 + 2.9694 * b + 0.4160 * b ** 2 - 0.5324 * b ** 3


print('Test statistics defined: kupiec_stat, christ_stat, kupiec_scalar, christ_scalar, hac_z, cv_fixed_b')


Test statistics defined: kupiec_stat, christ_stat, kupiec_scalar, christ_scalar, hac_z, cv_fixed_b


### Data-generating process

In [5]:
def viol_matrix(rng, dist, H, T, REPS, z90):
    """Violations of the 80% interval for H-day overlapping sums (normal or t5)."""
    if dist == 'normal':
        r = rng.standard_normal((REPS, T + H))
        thr = z90 * np.sqrt(H)
    else:
        r = rng.standard_t(5, (REPS, T + H))
        thr = np.quantile(np.abs(rng.standard_t(5, (200000, H)).sum(1)), 0.80)
    c = np.cumsum(r, axis=1)
    agg = c[:, H:] - c[:, :-H]
    return (np.abs(agg) > thr).astype(np.int8)


print('DGP function defined: viol_matrix')


DGP function defined: viol_matrix


---
## Table 1: Size Study (Normal DGP)

In [6]:
# Size study (Table 1)
print('Table 1: Size study (Normal DGP)')
ref1 = REF['table1_size_normal']
V = viol_matrix(rng, 'normal', H, T, REPS, z90)
for s in [1, 30, 60, 90]:
    exp = ref1['s'][s]
    Vs = V[:, ::s]
    rk = (kupiec_stat(Vs) < 0.05).mean() * 100
    rc = (christ_stat(Vs) < 0.05).mean() * 100
    check(f'Size s={s} Kupiec%', rk, exp['kupiec'], ref1['tolerance_pp'])
    check(f'Size s={s} Christoffersen%', rc, exp['christoffersen'], ref1['tolerance_pp'])
check('Empirical coverage (Normal)', 100 * (1 - V.mean()),
      ref1['empirical_coverage_pct'], ref1['empirical_coverage_tolerance_pp'])


Table 1: Size study (Normal DGP)
[PASS] Size s=1 Kupiec%: got=79.000 reference=79.000 (tol=3.0)
[PASS] Size s=1 Christoffersen%: got=100.000 reference=100.000 (tol=3.0)
[PASS] Size s=30 Kupiec%: got=14.900 reference=14.900 (tol=3.0)
[PASS] Size s=30 Christoffersen%: got=55.600 reference=55.600 (tol=3.0)
[PASS] Size s=60 Kupiec%: got=7.200 reference=7.200 (tol=3.0)
[PASS] Size s=60 Christoffersen%: got=5.500 reference=5.500 (tol=3.0)
[PASS] Size s=90 Kupiec%: got=5.400 reference=5.400 (tol=3.0)
[PASS] Size s=90 Christoffersen%: got=1.900 reference=1.900 (tol=3.0)
[PASS] Empirical coverage (Normal): got=80.224 reference=80.220 (tol=0.3)


True

---
## Table 2: Power Study

In [7]:
# Power study 2a: unconditional miscalibration (Table 2, rows 1-2)
print('Table 2: Power vs unconditional miscalibration')
ref2a = REF['table2_power_unconditional']
r = rng.standard_normal((REPS, T + H))
c = np.cumsum(r, axis=1)
agg = c[:, H:] - c[:, :-H]
for cov_true, z in [(0.70, stats.norm.ppf(0.85)), (0.90, stats.norm.ppf(0.95))]:
    thr = z * np.sqrt(H)
    V = (np.abs(agg) > thr).astype(np.int8)
    exp_row = ref2a['true_coverage'][cov_true]
    for s in [60, 90]:
        Vs = V[:, ::s]
        rk = (kupiec_stat(Vs) < 0.05).mean() * 100
        check(f'Power cov={cov_true} s={s} Kupiec%', rk, exp_row[s], ref2a['tolerance_pp'])


Table 2: Power vs unconditional miscalibration
[PASS] Power cov=0.7 s=60 Kupiec%: got=29.900 reference=30.200 (tol=4.0)
[PASS] Power cov=0.7 s=90 Kupiec%: got=22.400 reference=23.000 (tol=4.0)
[PASS] Power cov=0.9 s=60 Kupiec%: got=31.500 reference=32.600 (tol=4.0)
[PASS] Power cov=0.9 s=90 Kupiec%: got=23.200 reference=24.000 (tol=4.0)


In [8]:
# Power study 2b: genuine clustering (Table 2, rows 3-4)
print('Table 2: Power vs genuine clustering (Markov vol)')
ref2b = REF['table2_power_clustering']
p_stay = 0.98
s_hi, s_lo = 2.0, 0.6
states = np.zeros((REPS, T + H), dtype=np.int8)
u = rng.random((REPS, T + H))
states[:, 0] = (rng.random(REPS) < 0.5)
for t in range(1, T + H):
    stay = u[:, t] < p_stay
    states[:, t] = np.where(stay, states[:, t - 1], 1 - states[:, t - 1])
sig = np.where(states == 1, s_hi, s_lo)
r2 = rng.standard_normal((REPS, T + H)) * sig
c2 = np.cumsum(r2, axis=1)
agg2 = c2[:, H:] - c2[:, :-H]
thr2 = np.quantile(np.abs(agg2[:200].ravel()), 0.80)
V2 = (np.abs(agg2) > thr2).astype(np.int8)
check('Coverage under clustering DGP', 100 * (1 - V2.mean()), ref2b['coverage_pct'], ref2b['coverage_tolerance_pp'])
for s in [60, 90]:
    exp = ref2b['s'][s]
    Vs = V2[:, ::s]
    rc = (christ_stat(Vs) < 0.05).mean() * 100
    rk = (kupiec_stat(Vs) < 0.05).mean() * 100
    check(f'Power-clustering s={s} Christoffersen% (below nominal!)', rc, exp['christoffersen'], ref2b['tolerance_pp'])
    check(f'Power-clustering s={s} Kupiec%', rk, exp['kupiec'], ref2b['tolerance_pp'])


Table 2: Power vs genuine clustering (Markov vol)
[PASS] Coverage under clustering DGP: got=80.047 reference=80.200 (tol=0.5)
[PASS] Power-clustering s=60 Christoffersen% (below nominal!): got=6.500 reference=4.700 (tol=3.0)
[PASS] Power-clustering s=60 Kupiec%: got=8.600 reference=6.800 (tol=3.0)
[PASS] Power-clustering s=90 Christoffersen% (below nominal!): got=3.500 reference=2.800 (tol=3.0)
[PASS] Power-clustering s=90 Kupiec%: got=6.600 reference=6.500 (tol=3.0)


---
## Table 7: HAC/fixed-b Test

In [9]:
# HAC/fixed-b test (Table 7)
print('Table 7: HAC/fixed-b daily test')
ref7 = REF['table7_hac_fixed_b']
for Tn in [1000, 2000, 4000]:
    rr = rng.standard_normal((REPS, Tn + H))
    cc = np.cumsum(rr, axis=1)
    aa = cc[:, H:] - cc[:, :-H]
    Vn = (np.abs(aa) > z90 * np.sqrt(H)).astype(np.float64)
    z, L, n = hac_z(Vn, H)
    b = L / n
    cv = cv_fixed_b(b)
    rej = (np.abs(z) > cv).mean() * 100
    exp = ref7['T'][Tn]
    if exp is not None:
        check(f'HAC size T={Tn}', rej, exp, ref7['tolerance_pp'])
    else:
        print(f'  (T={Tn}, informational): rej={rej:.1f}% expected ~9-11%')


Table 7: HAC/fixed-b daily test
[PASS] HAC size T=1000: got=11.400 reference=13.000 (tol=3.0)
[PASS] HAC size T=2000: got=10.400 reference=11.200 (tol=3.0)
  (T=4000, informational): rej=9.9% expected ~9-11%


---
## Table 9: GARCH-VaR Empirical (BTC leg)

In [10]:
# GARCH-VaR empirical (Table 9, BTC leg)
import time
import warnings
from arch import arch_model
from tqdm.auto import tqdm


def garch_backtest(close, H, MIN_TRAIN, RETRAIN, z_q, label=None, verbose=True):
    """Walk-forward GARCH(1,1)-t backtest with H-day variance-scaling intervals, refit every RETRAIN days."""
    logc = np.log(close.clip(lower=1e-9))
    ret = 100 * logc.diff().dropna()
    r = ret.values
    n = len(r)
    tgt = (logc.shift(-H) - logc).reindex(ret.index).values
    viol = []
    t0 = time.time()
    params = None
    sig2 = None
    desc = f'GARCH backtest ({label})' if label else 'GARCH backtest'
    for i in tqdm(range(MIN_TRAIN, n - H), desc=desc, leave=False):
        if params is None or (i - MIN_TRAIN) % RETRAIN == 0:
            am = arch_model(r[:i], vol='GARCH', p=1, q=1, dist='t', mean='Constant')
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                res = am.fit(disp='off', show_warning=False)
            mu = float(res.params['mu'])
            om = float(res.params['omega'])
            a = float(res.params['alpha[1]'])
            b = float(res.params['beta[1]'])
            eps = r[:i] - mu
            s2 = np.empty(i)
            s2[0] = eps.var()
            for t in range(1, i):
                s2[t] = om + a * eps[t - 1] ** 2 + b * s2[t - 1]
            sig2 = s2[-1]
            params = (mu, om, a, b)
        else:
            mu, om, a, b = params
            sig2 = om + a * (r[i - 1] - mu) ** 2 + b * sig2
        mu_, om_, a_, b_ = params
        pers = min(a_ + b_, 0.998)
        s2_next = om_ + a_ * (r[i - 1] - mu_) ** 2 + b_ * sig2
        uncond = om_ / (1 - pers)
        s2_path = uncond + (pers ** np.arange(H)) * (s2_next - uncond)
        half = z_q * np.sqrt(s2_path.sum()) / 100
        center = mu_ * H / 100
        if np.isnan(tgt[i]):
            continue
        viol.append(1 if abs(tgt[i] - center) > half else 0)
    v = np.array(viol, dtype=int)
    if verbose and label is not None:
        print(f'{label}: OOS={len(v):,} in {time.time() - t0:.0f}s | coverage={100 * (1 - v.mean()):.1f}%')
    return v


try:
    ALPHA = 0.20
    MIN_TRAIN = 700
    RETRAIN = 90
    ref_garch = REF['garch_btc']

    df = pd.read_csv(BTC_CSV, parse_dates=['Date'], index_col='Date').sort_index()
    df.columns = [c.capitalize() for c in df.columns]
    v = garch_backtest(df['Close'], H, MIN_TRAIN, RETRAIN, z90)
    print('=== Table 9: GARCH BTC ===')
    check('GARCH BTC coverage%', 100 * (1 - v.mean()),
          ref_garch['coverage_pct']['value'], ref_garch['coverage_pct']['tolerance_pp'])
    check('GARCH BTC Kupiec s=30', kupiec_scalar(v[::30], ALPHA),
          ref_garch['kupiec_s30']['value'], ref_garch['kupiec_s30']['tolerance'])
    check('GARCH BTC Kupiec s=90', kupiec_scalar(v[::90], ALPHA),
          ref_garch['kupiec_s90']['value'], ref_garch['kupiec_s90']['tolerance'])
    check('GARCH BTC Christoffersen s=90', christ_scalar(v[::90], ALPHA),
          ref_garch['christ_s90']['value'], ref_garch['christ_s90']['tolerance'])
except FileNotFoundError:
    print('[NOTE]  data/btc_binance_daily.csv not found -- skipping the GARCH check.')
    print('    (The Size/Power/HAC checks above do not depend on this file.)')


/home/mykola/BTC_q_pred/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
                                                                               

=== Table 9: GARCH BTC ===
[PASS] GARCH BTC coverage%: got=90.089 reference=90.100 (tol=0.5)
[PASS] GARCH BTC Kupiec s=30: got=0.007 reference=0.007 (tol=0.02)
[PASS] GARCH BTC Kupiec s=90: got=0.115 reference=0.115 (tol=0.05)
[PASS] GARCH BTC Christoffersen s=90: got=0.238 reference=0.238 (tol=0.08)


---
## Final Verdict

In [11]:
# -- FINAL VERDICT -------------------------------------------------------------
n_ok = sum(1 for *_, ok in PASS_LOG if ok)
n_tot = len(PASS_LOG)
print(f'Passed: {n_ok}/{n_tot} checks\n')
if n_ok == n_tot:
    print('[OK] All reference values are reproduced within tolerance.')
else:
    print('[WARN] Discrepancies found -- review entries marked [FAIL] before submission.')
    for label, got, exp, ok in PASS_LOG:
        if not ok:
            print(f'   [FAIL] {label}: got={got:.3f} vs reference={exp:.3f}')

Passed: 24/24 checks

[OK] All reference values are reproduced within tolerance.
